# Garmin Connect API Exploration

Interactive testing ground for the `garminconnect` dependency ahead of building `src/ingestion/garmin_client.py`.

**Goals**

1. Confirm authentication works using credentials from `src/core/config.py` (backed by `.env`).
2. Focus area: downloading **FIT files** for activities and wellness/monitoring data (per the architecture doc's Bronze-layer "FIT vs JSON" investigation).
3. Survey the rest of the 150+ method API surface so we know what's available for later ingestion work.

**Before running:**

- Copy [`.env.template`](../.env.template) to `.env` at the repo root and fill in `GARMIN__EMAIL` / `GARMIN__PASSWORD`.
- Cells in this notebook make **live calls to your real Garmin Connect account** and will write real personal health/activity data to `data/_scratch_garmin_test/` (git-ignored). Nothing here is committed to version control.
- On first login you may be prompted for an MFA code inline (`input()`); afterwards, OAuth tokens are cached (`GARMIN__TOKENSTORE_PATH`, default `~/.garminconnect`) so subsequent runs skip credential login.


In [8]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not locate repository root (pyproject.toml not found).")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"No .env file found at {env_path}. Copy .env.template to .env and fill in "
        "your Garmin credentials before running this notebook."
    )

from src.core.config import get_settings

settings = get_settings()

# Scratch area for this notebook's downloads only — never used by the real
# ingestion pipeline (which will write to settings.data.bronze_dir).
SCRATCH_DIR = REPO_ROOT / "data" / "_scratch_garmin_test"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:     {REPO_ROOT}")
print(f"Garmin account: {settings.garmin.email is not None and 'CONFIGURED' or 'MISSING'}")
print(f"Token cache:    {settings.garmin.tokenstore_path}")
print(f"Scratch dir:    {SCRATCH_DIR}")


Repo root:     c:\Users\gabri\workspace\strider
Garmin account: CONFIGURED
Token cache:    C:\Users\gabri\.garminconnect
Scratch dir:    c:\Users\gabri\workspace\strider\data\_scratch_garmin_test


## Authentication

`Garmin.login()` uses the same mobile SSO flow as the Android app. `prompt_mfa` is only invoked if
your account has MFA enabled. Passing a `tokenstore` path lets the library cache tokens on disk and
auto-refresh them, so re-running this notebook later won't require a fresh login.


**⚠️ Garmin's SSO login endpoint is IP-rate-limited.** A fresh credential login (no cached tokens

yet) can trigger a `429 Too Many Requests` on the very first attempt, even before you've donetypically IP-based and needs time to clear regardless of what this code does.

anything wrong — the library tries several login strategies in a row, which compounds the risk.If you hit the cooldown, **wait** rather than passing `force=True` immediately — Garmin's block is

Re-running a failed login immediately makes this worse, not better.

- Once tokens are cached successfully, subsequent runs load them from disk with no fresh SSO call.

To avoid this, authentication goes through `src/ingestion/garmin_client.py`, which:  `rate_limit.login_cooldown_seconds` in `src/core/config.py` (default 1 hour).

- Skips `login()` entirely if this session is already authenticated (safe to re-run this cell).  attempts (even across kernel restarts) until the cooldown elapses — see
- Persists a **cooldown marker** next to the token cache after any 429, and refuses further login

In [2]:
from garminconnect import GarminConnectAuthenticationError, GarminConnectTooManyRequestsError

from src.ingestion.garmin_client import GarminAuthCooldownError, build_client, login

client = build_client(settings)

try:
    login(client, settings)
    print(
        "Authenticated"
        + (" (session already active)." if client.client.is_authenticated else ".")
    )
except GarminAuthCooldownError as e:
    print(f"Login blocked by cooldown from a prior rate-limit: {e}")
    raise
except GarminConnectAuthenticationError as e:
    print(f"Authentication failed — check GARMIN__EMAIL / GARMIN__PASSWORD in .env: {e}")
    raise
except GarminConnectTooManyRequestsError as e:
    print(f"Rate limited by Garmin Connect — cooldown recorded, wait before retrying: {e}")
    raise


mobile+cffi returned 429: GarminConnectTooManyRequestsError: Mobile login returned 429 — IP rate limited by Garmin
mobile+requests returned 429: GarminConnectTooManyRequestsError: Mobile login returned 429 — IP rate limited by Garmin


Authenticated (session already active).


## Profile & daily stats (JSON)

Quick sanity checks that the session works, using cheap, low-volume endpoints.


In [10]:
from datetime import date

today = date.today().isoformat()

print("Full name:  ", client.get_full_name())
print("Unit system:", client.get_unit_system())

stats = client.get_stats(today)
print(f"\nToday ({today}):")
print(f"  Steps:        {stats.get('totalSteps')}")
print(f"  Calories:     {stats.get('totalKilocalories')}")
print(f"  Resting HR:   {stats.get('restingHeartRate')}")
print(f"  Stress avg:   {stats.get('averageStressLevel')}")


Full name:   Gabriele Di Corato
Unit system: metric

Today (2026-09-26):
  Steps:        6836
  Calories:     1315.0
  Resting HR:   57
  Stress avg:   37


## Activities & FIT downloads (primary focus)

`download_activity(activity_id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL)` returns the
**original binary FIT file, zipped**, exactly as recorded by the device — this is what
`src/processing/parser_fit.py` (via `fitdecode`) should consume for the Silver layer, per the
architecture doc's directive to prefer FIT over JSON where available. Other formats (`TCX`, `GPX`,
`KML`, `CSV`) are lossy derivatives and not used by the pipeline.


In [11]:
total = client.count_activities()
print(f"Total activities on account: {total}\n")

recent = client.get_activities(start=0, limit=5)
for act in recent:
    activity_type = act.get("activityType", {}).get("typeKey", "?")
    print(
        f"{act['activityId']:>12}  {act.get('startTimeLocal', '?'):>20}  "
        f"{act.get('activityName', '?'):<30}  {activity_type}"
    )


Total activities on account: 1245

 24501751461   2026-09-26 10:06:06  Lodi Walking                    walking
 24501749482   2026-09-26 00:20:07  Lodi Walking                    walking
 24484858072   2026-09-24 18:52:10  Lodi Walking                    walking
 24480686682   2026-09-24 12:53:35  Lodi - Speed Repeats            running
 24472589847   2026-09-23 18:45:41  Lodi Walking                    walking


In [ ]:
import zipfile

if not recent:
    print("No activities on the account to download.")
else:
    activity_id = recent[0]["activityId"]
    activity_name = recent[0].get("activityName", "unnamed")

    fit_zip_bytes = client.download_activity(
        activity_id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL
    )
    zip_path = SCRATCH_DIR / f"activity_{activity_id}.zip"
    zip_path.write_bytes(fit_zip_bytes)
    print(f"'{activity_name}' -> {len(fit_zip_bytes):,} bytes saved to {zip_path}")

    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"  contains: {name} ({size:,} bytes)")


NameError: name 'Garmin' is not defined

## Wellness / monitoring data

Most day-level wellness metrics (`get_heart_rates`, `get_sleep_data`, `get_stress_data`, `get_hrv_data`,
`get_body_battery`, `get_spo2_data`, `get_respiration_data`, `get_training_status`, ...) are only
exposed as **JSON**, not FIT — per the architecture doc, these should land in Bronze as raw JSON and
then be shredded with Polars into Silver Parquet, rather than parsed as binary.

`download_health_snapshot(date)` is the one exception worth checking: despite its name, the endpoint
it calls is `/download-service/files/wellness/{date}` — the same family as the activity FIT download
route (`/download-service/files/activity/{id}`). We download it below to see whether it actually
contains a **binary monitoring FIT file** for the day (steps/HR/stress epochs), which would let us
prefer binary parsing for wellness data too, as the architecture doc directs us to investigate.


In [ ]:
wellness_calls = {
    "get_heart_rates": client.get_heart_rates,
    "get_sleep_data": client.get_sleep_data,
    "get_stress_data": client.get_stress_data,
    "get_hrv_data": client.get_hrv_data,
    "get_body_battery": client.get_body_battery,
    "get_spo2_data": client.get_spo2_data,
    "get_respiration_data": client.get_respiration_data,
    "get_training_status": client.get_training_status,
    "get_training_readiness": client.get_training_readiness,
}

for name, fn in wellness_calls.items():
    try:
        result = fn(today)
        if isinstance(result, dict):
            shape = f"dict with keys: {list(result.keys())[:8]}"
        elif isinstance(result, list):
            shape = f"list of {len(result)} item(s)"
        else:
            shape = type(result).__name__
        print(f"{name:<24} -> {shape}")
    except Exception as e:
        print(f"{name:<24} -> ERROR: {e}")


In [ ]:
from datetime import timedelta

from garminconnect import GarminConnectNotFoundError

# Today's snapshot may not be finalized yet server-side; fall back to yesterday.
candidate_dates = [today, (date.today() - timedelta(days=1)).isoformat()]

snapshot_bytes = None
snapshot_date = None
for d in candidate_dates:
    try:
        snapshot_bytes = client.download_health_snapshot(d)
        snapshot_date = d
        break
    except (GarminConnectNotFoundError, GarminConnectConnectionError) as e:
        print(f"No snapshot for {d}: {e}")

if snapshot_bytes is None:
    print("No health/wellness snapshot available for the recent dates tried.")
else:
    snapshot_path = SCRATCH_DIR / f"wellness_{snapshot_date}.zip"
    snapshot_path.write_bytes(snapshot_bytes)
    print(f"Saved {len(snapshot_bytes):,} bytes -> {snapshot_path}")

    with zipfile.ZipFile(snapshot_path) as zf:
        names = zf.namelist()
        print("Contents:", names)
        for name in names:
            marker = " <-- binary FIT!" if name.lower().endswith(".fit") else ""
            print(f"  - {name} ({zf.getinfo(name).file_size:,} bytes){marker}")


## Broader API surface

`garminconnect` exposes 150+ methods on the `Garmin` class. Rather than hardcoding a list that will
drift as the dependency is upgraded, introspect the installed version directly and bucket the public
methods by theme so we know what's available for future ingestion work beyond activities/wellness
(goals, gear, training plans, golf, nutrition, menstrual health, etc.).


In [ ]:
import inspect

public_methods = sorted(
    name
    for name, member in inspect.getmembers(Garmin, predicate=inspect.isfunction)
    if not name.startswith("_")
)
print(f"Total public methods on Garmin: {len(public_methods)}")

keyword_categories = {
    "activities": ("activit",),
    "wellness/health": (
        "sleep", "stress", "hrv", "spo2", "respiration", "battery",
        "heart_rate", "heartrate", "hydration", "blood_pressure", "rhr",
    ),
    "training/fitness metrics": (
        "training", "endurance", "race", "fitnessage", "hill_score",
        "max_metrics", "lactate", "ftp", "cycling_ftp", "zones",
    ),
    "body composition/weight": ("weigh", "body_composition", "body_battery"),
    "gear/device": ("gear", "device"),
    "workouts/training plans": ("workout", "training_plan"),
    "goals/badges/challenges": ("goal", "badge", "challenge", "record"),
    "golf": ("golf",),
    "menstrual/pregnancy": ("menstrual", "pregnancy"),
    "nutrition": ("nutrition",),
    "account/system": ("login", "logout", "profile", "graphql", "download", "connectapi", "typed", "reload"),
}

categorized: set[str] = set()
categories: dict[str, list[str]] = {}
for label, keywords in keyword_categories.items():
    matches = [m for m in public_methods if any(k in m for k in keywords)]
    categories[label] = matches
    categorized.update(matches)
categories["other"] = [m for m in public_methods if m not in categorized]

for label, methods in categories.items():
    print(f"\n{label} ({len(methods)})")
    for m in methods:
        print(f"  - {m}")


## Findings & next steps

- **Activities → FIT**: `download_activity(id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL)` gives the
  original device-recorded FIT file (zipped). This is the Bronze payload for activities; `parser_fit.py`
  will `fitdecode` it directly into Silver records.
- **Wellness/monitoring → mostly JSON**: `get_heart_rates`, `get_sleep_data`, `get_stress_data`,
  `get_hrv_data`, `get_body_battery`, `get_spo2_data`, `get_respiration_data`, `get_training_status`, etc.
  only return JSON — check the cell output above for whether `download_health_snapshot` also yielded a
  binary `.fit` file. If it did, prefer that route for wellness Bronze storage per the architecture
  doc; if not (e.g. it's an ECG-only "Health Snapshot" rather than general monitoring data), fall back
  to the documented pattern: save raw JSON to Bronze, then shred with Polars into Silver.
- **Everything else** (goals/badges, gear, training plans, golf, nutrition, menstrual health) is
  available but out of scope for the current roadmap step — revisit once Bronze→Silver ingestion for
  activities and wellness is idempotent and operational.

Next: build `src/ingestion/garmin_client.py` (thin adapter wrapping the calls exercised above) and
`src/ingestion/ledger.py` (hash/timestamp tracking per `activity_id` / date so re-syncs are safe).
